# 141. Linked List Cycle
**Difficulty:** 🟢 Easy · **Topic:** Linked List · **LeetCode:** https://leetcode.com/problems/linked-list-cycle/

## 💡 Concepts

**Core concept(s):** Detect a loop with a **hash set** of seen nodes, or **fast & slow** pointers.

**Why it applies here:** If the list loops, a fast pointer (2 steps) will eventually lap and meet a slow pointer (1 step) inside the loop. If there's no loop, fast simply reaches the end. The set version just remembers every node and spots a repeat.

**Key intuition:** Two runners on a circular track always meet; on a straight track the fast one just finishes.

---

### 📚 What is a Linked List?
A **linked list** is a chain of nodes; each node holds a value and a pointer to the **next** node. Unlike an array there is no index — you can only walk forward from the head.
- **In Python:** a small `ListNode` class with `.val` and `.next`.

### 📚 Fast & Slow Pointers
Two pointers moving at different speeds: **slow** one step, **fast** two. They meet inside a loop (cycle detection) and the slow one lands on the middle when fast reaches the end.
- **Complexity:** one pass, **O(1)** extra space.

---

**Prerequisite knowledge:**
- Node identity in a set.
- Fast/slow pointers.

## 📝 Problem

Return `True` if the linked list contains a cycle.

> Two approaches: hash set `O(n)` space and Floyd's fast/slow `O(1)` space.

In [ ]:
from typing import Optional, List

class ListNode:
    """A node in a singly linked list: a value plus a link to the next node."""
    def __init__(self, val=0, next=None):
        self.val = val                     # the value stored at this node
        self.next = next                   # link to the next node (None at the end)

def build_list(vals):
    """Turn a Python list into a linked list; return its head."""
    dummy = ListNode(); cur = dummy        # dummy node avoids special-casing the first node
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    """Turn a linked list back into a Python list (handy for printing / assertions)."""
    out = []
    while head:
        out.append(head.val); head = head.next
    return out

def build_cycle(vals, pos):
    """Build a list; if pos >= 0, link the tail back to node at index pos (a cycle)."""
    nodes = [ListNode(v) for v in vals]
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]       # chain the nodes together
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]        # make the last node point back -> a loop
    return nodes[0] if nodes else None

### Approach 1 — Hash Set (worst on memory)

**Idea:** Walk the list, remembering every node. A repeat means a cycle.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
def has_cycle_set(head: Optional[ListNode]) -> bool:
    seen = set()                           # nodes we've already visited
    while head:
        if head in seen:                   # met the same node twice -> there's a loop
            return True
        seen.add(head)                     # remember this node
        head = head.next
    return False                           # reached the end -> no loop

### Approach 2 — Fast & Slow (optimal)

**Idea:** Move slow by 1 and fast by 2. If they ever meet, there's a cycle; if fast falls off the end, there isn't.

**Time:** `O(n)`. **Space:** `O(1)`.

In [ ]:
def has_cycle_floyd(head: Optional[ListNode]) -> bool:
    slow = fast = head                     # two runners starting together
    while fast and fast.next:              # fast needs two nodes ahead to step
        slow = slow.next                   # slow moves 1 step
        fast = fast.next.next              # fast moves 2 steps
        if slow is fast:                   # they met -> the fast one lapped inside a loop
            return True
    return False                           # fast fell off the end -> no loop

In [ ]:
# Correctness check
tests = [([3,2,0,-4],1,True), ([1,2],0,True), ([1],-1,False), ([],-1,False)]
for vals, pos, exp in tests:
    a = has_cycle_set(build_cycle(vals, pos))
    b = has_cycle_floyd(build_cycle(vals, pos))
    print(f"{vals}, pos={pos} -> set={a}, floyd={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio when `n` → `2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_list(list(range(n))),)   # no cycle -> both scan the whole list
solutions = {
    "hash set O(n) space O(n)": has_cycle_set,
    "floyd    O(n) space O(1)": has_cycle_floyd,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Fast/slow pointers:** detect loops and find middles with O(1) memory.
- **Signal:** "cycle / loop", "does it repeat", "find the middle".
- **Related problems:** Linked List Cycle II (find start), Happy Number, Find the Duplicate Number.
- **Common pitfalls:** (1) checking `fast` and `fast.next` before stepping; (2) comparing values instead of node identity.